In [1]:
"""
==============================================================================
Exchange-Bias / Magnetic-Nanoparticle ML Benchmark
==============================================================================
Reproduces the central result of our analysis: a comparison of
    (a) standard shuffled K-Fold cross-validation (what is normally reported)
    (b) GroupKFold cross-validation, grouped by material "formula"
        (no material appears in both train and test in any fold -- this is
        the honest estimate of how the model performs on a genuinely new,
        unseen material)
across five magnetic targets, on two feature sets (baseline vs enhanced).

--------------------------------------------------------------------------
HOW TO RUN
--------------------------------------------------------------------------
1. Put this script in the same folder as these three files (from your
   project -- no changes needed to them):
       exchange_bias_baseline.csv
       exchange_bias_enhanced.csv
       final_data_exchange_bias.csv

2. Install dependencies (one time):
       pip install pandas numpy scikit-learn

3. Run:
       python run_benchmark.py

4. Output:
       - Full results table printed to the console
       - Saved to benchmark_results.csv (main leaky-vs-honest comparison)
       - Saved to benchmark_best_tuned.csv (best tuned model per target)

Expect this to take a few minutes on a laptop CPU (no GPU needed).
==============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.preprocessing import MinMaxScaler


# ==============================================================================
# STEP 1 -- Load data
# ==============================================================================
print("STEP 1: Loading data...")

baseline = pd.read_csv("exchange_bias_baseline.csv")
enhanced = pd.read_csv("exchange_bias_enhanced.csv")
final = pd.read_csv("final_data_exchange_bias.csv")

assert len(baseline) == len(enhanced) == len(final), (
    "Row counts don't match -- these three files must be the same rows "
    "with different columns (as produced by data_pros1.ipynb -> feat_eng_new.ipynb)."
)

# 'formula' identifies the specific core-shell material (e.g. "Co-CoO").
# This is what we group by for the honest evaluation.
formula = final["formula"]

print(f"  Loaded {len(final)} rows, {formula.nunique()} unique materials "
      f"(this repetition is the source of the leakage we're testing for).")


# ==============================================================================
# STEP 2 -- Define targets and known data issues
# ==============================================================================
print("\nSTEP 2: Defining targets and fixing known data issues...")

TARGETS = {
    "Exchange Bias":            "exc_bias_oe",
    "Coercivity":                "coer_oe",
    "Saturation Magnetization":  "sat_em_g",
    "Curie Temperature":         "Tc",
    "Remanence":                 "mr (emu/g)",
}

# These four targets are heavily right-skewed (skew > 1.5); log1p stabilizes
# variance before modeling. Curie Temperature is left untransformed (already
# close to symmetric).
LOG_TARGETS = {"exc_bias_oe", "coer_oe", "sat_em_g", "mr (emu/g)"}

# Data-quality fix: 11 rows have a saturation-magnetization unit error
# (values up to 106,600 emu/g -- physically implausible for these material
# classes, real max is ~250 emu/g). We exclude them ONLY when sat_em_g is
# the target (they stay in the data for every other target).
SAT_OUTLIER_ROWS = final.index[final["sat_em_g"] > 1000]
print(f"  Found {len(SAT_OUTLIER_ROWS)} saturation-magnetization outlier rows "
      f"(unit-error block) -- will be excluded when sat_em_g is the target.")


# ==============================================================================
# STEP 3 -- Models
# ==============================================================================
MODELS = {
    "RandomForest":         RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42, max_iter=150),
}


# ==============================================================================
# STEP 4 -- Core evaluation function
# ==============================================================================
def evaluate(feature_df, target_col, group_series, drop_idx=None,
             log_target=False, n_folds=5):
    """
    Runs both shuffled KFold and GroupKFold-by-formula for every model in
    MODELS, and returns a tidy results DataFrame.
    """
    d = feature_df.copy()
    g = group_series.copy()
    if drop_idx is not None:
        d = d.drop(index=drop_idx)
        g = g.drop(index=drop_idx)

    X = d.drop(columns=[target_col])
    y = d[target_col].values
    if log_target:
        y = np.log1p(y)

    X_scaled = MinMaxScaler().fit_transform(X)

    splitters = {
        "shuffled_kfold (leaky)":        (KFold(n_splits=n_folds, shuffle=True, random_state=42), {}),
        "groupkfold_by_formula (honest)": (GroupKFold(n_splits=n_folds), dict(groups=g.values)),
    }

    rows = []
    for split_name, (splitter, split_kwargs) in splitters.items():
        for model_name, model in MODELS.items():
            cv = cross_validate(
                model, X_scaled, y, cv=splitter,
                scoring=["r2"], n_jobs=-1, **split_kwargs,
            )
            rows.append({
                "split": split_name,
                "model": model_name,
                "r2_mean": round(cv["test_r2"].mean(), 3),
                "r2_std": round(cv["test_r2"].std(), 3),
            })
    return pd.DataFrame(rows)


# ==============================================================================
# STEP 5 -- Run the main leaky-vs-honest comparison for every target
# ==============================================================================
print("\nSTEP 5: Running shuffled-KFold vs GroupKFold for all 5 targets "
      "x 2 feature sets (this is the main result -- takes a few minutes)...")

all_results = []
for target_name, target_col in TARGETS.items():
    log_t = target_col in LOG_TARGETS
    drop_idx = SAT_OUTLIER_ROWS if target_col == "sat_em_g" else None

    for feature_set_name, feature_df in [("baseline", baseline), ("enhanced", enhanced)]:
        print(f"  -> {target_name} / {feature_set_name}")
        res = evaluate(feature_df, target_col, formula, drop_idx=drop_idx, log_target=log_t)
        res["target"] = target_name
        res["feature_set"] = feature_set_name
        all_results.append(res)

results_df = pd.concat(all_results, ignore_index=True)
results_df = results_df[["target", "feature_set", "split", "model", "r2_mean", "r2_std"]]
results_df.to_csv("benchmark_results.csv", index=False)

print("\n" + "=" * 78)
print("MAIN RESULT: shuffled KFold (leaky) vs GroupKFold by formula (honest)")
print("=" * 78)
print(results_df.to_string(index=False))
print(f"\nSaved to benchmark_results.csv")


# ==============================================================================
# STEP 6 -- Best tuned model per target (GroupKFold only, baseline features)
# ==============================================================================
print("\nSTEP 6: Finding the best honest (GroupKFold) model per target "
      "with a small model/hyperparameter sweep...")

TUNED_MODELS = {
    "RandomForest_default": RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    "RandomForest_deep":    RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2,
                                                   random_state=42, n_jobs=-1),
    "ExtraTrees":           ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "HGB_default":          HistGradientBoostingRegressor(random_state=42, max_iter=150),
    "HGB_regularized":      HistGradientBoostingRegressor(random_state=42, max_iter=200, learning_rate=0.05,
                                                            max_leaf_nodes=15, l2_regularization=1.0),
}

def evaluate_groupkfold_only(feature_df, target_col, group_series, models,
                              drop_idx=None, log_target=False, n_folds=5):
    d = feature_df.copy()
    g = group_series.copy()
    if drop_idx is not None:
        d = d.drop(index=drop_idx)
        g = g.drop(index=drop_idx)
    X = d.drop(columns=[target_col])
    y = d[target_col].values
    if log_target:
        y = np.log1p(y)
    X_scaled = MinMaxScaler().fit_transform(X)

    rows = []
    for model_name, model in models.items():
        cv = cross_validate(model, X_scaled, y, cv=GroupKFold(n_splits=n_folds),
                             groups=g.values, scoring=["r2"], n_jobs=-1)
        rows.append({"model": model_name,
                      "r2_mean": round(cv["test_r2"].mean(), 3),
                      "r2_std": round(cv["test_r2"].std(), 3)})
    return pd.DataFrame(rows).sort_values("r2_mean", ascending=False)

best_rows = []
for target_name, target_col in TARGETS.items():
    log_t = target_col in LOG_TARGETS
    drop_idx = SAT_OUTLIER_ROWS if target_col == "sat_em_g" else None
    print(f"  -> {target_name}")
    res = evaluate_groupkfold_only(baseline, target_col, formula, TUNED_MODELS,
                                    drop_idx=drop_idx, log_target=log_t)
    best = res.iloc[0]
    best_rows.append({"target": target_name, "best_model": best["model"],
                       "r2_mean": best["r2_mean"], "r2_std": best["r2_std"]})

best_df = pd.DataFrame(best_rows)
best_df.to_csv("benchmark_best_tuned.csv", index=False)

print("\n" + "=" * 78)
print("BEST HONEST (GroupKFold) RESULT PER TARGET, AFTER TUNING")
print("=" * 78)
print(best_df.to_string(index=False))
print(f"\nSaved to benchmark_best_tuned.csv")

print("\nDone. Compare your printed numbers to the summary table in the "
      "leakage_aware_benchmark_summary.md document -- they should match "
      "closely (small differences are expected from library-version "
      "differences in RandomForest/HistGradientBoosting internals).")


STEP 1: Loading data...
  Loaded 979 rows, 200 unique materials (this repetition is the source of the leakage we're testing for).

STEP 2: Defining targets and fixing known data issues...
  Found 11 saturation-magnetization outlier rows (unit-error block) -- will be excluded when sat_em_g is the target.

STEP 5: Running shuffled-KFold vs GroupKFold for all 5 targets x 2 feature sets (this is the main result -- takes a few minutes)...
  -> Exchange Bias / baseline
  -> Exchange Bias / enhanced
  -> Coercivity / baseline
  -> Coercivity / enhanced
  -> Saturation Magnetization / baseline
  -> Saturation Magnetization / enhanced
  -> Curie Temperature / baseline
  -> Curie Temperature / enhanced
  -> Remanence / baseline
  -> Remanence / enhanced

MAIN RESULT: shuffled KFold (leaky) vs GroupKFold by formula (honest)
                  target feature_set                          split                model  r2_mean  r2_std
           Exchange Bias    baseline         shuffled_kfold (leaky)  